<a href="https://colab.research.google.com/github/Bilal-data/arabic-spam-classifier/blob/main/arabic_spam_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Arabic Spam / Ham Tweet Classifier
TF-IDF + Logistic Regression on **13,240 Arabic tweets** ("Dataset of Arabic Spam and Ham Tweets").

This notebook is the larger-dataset stage of the project. The earlier stage used 50 hand-written messages, where results swung a lot between splits (see the README).

## 1. Load the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

f = "/content/drive/MyDrive/Dataset of Arabic Spam and Ham Tweets (1).xlsx"
tw = pd.read_excel(f)

print(tw.shape)
print(tw.columns.tolist())

(13240, 16)
['Date', 'Time', 'Date Time', 'URL', 'Tweet Text', 'Cleaned Text', 'User Name', 'Location', 'Replied Tweet ID ', 'Replied Tweet User ID', 'Replied Tweet User name', 'Coordinates', 'Retweet Count', 'Favorite Count', 'Favorited', 'Label']


## 2. Check the classes
The data is imbalanced: far more Ham than Spam.

In [ ]:
print(tw["Label"].value_counts())
print(tw[["Tweet Text", "Label"]].isna().sum())

Label
Ham     11299
Spam     1941
Name: count, dtype: int64
Tweet Text    0
Label         0
dtype: int64


## 3. Train / test split
80/20, stratified so both parts keep the same Ham/Spam ratio.

In [ ]:
from sklearn.model_selection import train_test_split

data = tw[["Tweet Text", "Label"]]

X_train, X_test, y_train, y_test = train_test_split(
    data["Tweet Text"], data["Label"],
    test_size=0.2, random_state=42, stratify=data["Label"]
)

print(len(X_train), len(X_test))
print(y_test.value_counts())

10592 2648
Label
Ham     2260
Spam     388
Name: count, dtype: int64


## 4. Model: TF-IDF + Logistic Regression

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)

print("Accuracy:", pipe.score(X_test, y_test))

Accuracy: 0.9818731117824774


## 5. Precision and recall per class
Accuracy alone is not enough on imbalanced data, so we look at the Spam class separately.

In [ ]:
from sklearn.metrics import classification_report

pred = pipe.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

         Ham       0.98      1.00      0.99      2260
        Spam       1.00      0.88      0.93       388

    accuracy                           0.98      2648
   macro avg       0.99      0.94      0.96      2648
weighted avg       0.98      0.98      0.98      2648



## 6. Baseline: what does a model that always says "Ham" score?
This is the number the model must beat for the accuracy to mean anything.

In [ ]:
print("Baseline (always Ham):", round((y_test == "Ham").mean(), 4))

Baseline (always Ham): 0.8535


In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, data["Tweet Text"], data["Label"], cv=cv)

print(scores)
print("Mean:", scores.mean())

[0.9826284  0.98074018 0.98036254 0.98187311 0.98225076]
Mean: 0.981570996978852


In [ ]:
from sklearn.metrics import make_scorer, recall_score

spam_recall = make_scorer(recall_score, pos_label="Spam")
r = cross_val_score(pipe, data["Tweet Text"], data["Label"], cv=cv, scoring=spam_recall)

print(r)
print("Mean:", r.mean())

[0.8814433  0.8685567  0.86597938 0.87628866 0.87917738]
Mean: 0.8742890838258288


## Cross-validation results (5-fold, stratified)
- Accuracy: 0.9804 to 0.9826 across folds, mean 0.9816 (very stable)
- Spam recall: 0.866 to 0.881 across folds, mean 0.874 (about 1 in 8 spam tweets is missed)

## Limitations
- Main result uses one 80/20 split; the 5-fold CV above shows it is stable
- Tweets, not SMS or email
- Imbalanced classes; some spam is still missed (recall about 0.87)
- No hyperparameter tuning
- Not production-ready